In [1]:
import pandas as pd
df=pd.read_csv(r"IMDB Dataset.csv")

In [3]:
print(df.head())
print(df.info())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None
positive    25000
negative    25000
Name: sentiment, dtype: int64


In [5]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [7]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [9]:
def preprocess_text(text):
    text = text.lower()  
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  
    text = re.sub(r'[^a-zA-Z]', ' ', text)  
    words = text.split() 
    words = [w for w in words if w not in stop_words]  
    words = [stemmer.stem(w) for w in words]  
    return " ".join(words)

In [11]:
df['cleaned_review'] = df['review'].apply(preprocess_text)
print(df[['review','cleaned_review']].head())

                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                      cleaned_review  
0  one review mention watch oz episod hook right ...  
1  wonder littl product br br film techniqu unass...  
2  thought wonder way spend time hot summer weeke...  
3  basic famili littl boy jake think zombi closet...  
4  petter mattei love time money visual stun film...  


In [13]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['cleaned_review']).toarray()

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned_review']).toarray()

In [17]:
y = df['sentiment']

In [25]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [26]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [27]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

In [28]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred, model_name):
    print(f"--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print()

In [35]:
evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")

--- Logistic Regression ---
Accuracy: 0.8861
Precision: 0.8761574074074074
Recall: 0.9013693193093868
F1 Score: 0.88858456421794

--- Naive Bayes ---
Accuracy: 0.8516
Precision: 0.8483245149911817
Recall: 0.8590990275848382
F1 Score: 0.8536777755866692

--- Decision Tree ---
Accuracy: 0.7179
Precision: 0.7237691686844229
Recall: 0.711847588807303
F1 Score: 0.7177588794397198



In [ ]:
# 7
'''
TF-IDF performed better than BoW because it gives importance to meaningful words.
Logistic Regression gave highest accuracy because it handles text data well
Naive Bayes was fast but slightly less accurate
Decision Tree overfitted and gave lower performance

In [ ]:
Raw Text - Cleaning -Tokenization - Stopword Removal - Stemming - TF-IDF - ML Model - Evaluation